## Phase 2, Step 1: which ownership links the 50% calculation will use

**The goal of Phase 2:** under the US "50% rule", a company is treated as sanctioned if sanctioned owners together hold 50% or more of it, directly or through other companies they control. To calculate that, we first need a clean list of who owns whom.

### What one row means
Each row in silver.ownership is one ownership link: **one owner holds a share of one company.**
Example: "Company A owns 60% of Company B."

| Column | Plain meaning | Example |
|---|---|---|
| ownership_id | the ID of this ownership link | ru-6096a14f... |
| owner_ref | who owns, exactly as the source wrote it | ru-inn-7707329152 |
| asset_ref | what is owned, exactly as the source wrote it | ru-inn-4011017003 |
| owner_id | who owns, using our standard ID | ru-inn-7707329152 |
| asset_id | what is owned, using our standard ID | ru-inn-4011017003 |

**Standard ID (canonical_id):** the same company often appears in several sources under different IDs. OpenSanctions recognises these as one company and gives them one shared ID. Example: Mekhatronika appears as ru-inn-3702686347 (Russian register) and permid-5086648493 (another source), and both map to NK-R5MBqWzCxd5iUvZzYTctC3. We use this shared ID everywhere, so each company appears once.

- Several owners of one company, or one owner of several companies, show up as several rows.
- owner_id = asset_id would mean a company owns itself. 247 such rows exist; they are data errors and are left out.

### What we checked
- Translating to the standard ID changed nothing: on all 206,390 links, the source already used the standard ID. We keep the step anyway, in case future data doesn't.
- Every company that owns or is owned has a matching row in silver.legal_entity (0 missing). The only exceptions are things that aren't companies: people, vessels, aircraft and other assets.

### Profiling the current links (Nov 1)
| Measure | Result |
|---|---|
| Current links (excluding companies owning themselves) | 110,079 |
| Distinct owner -> company pairs | 109,513 |
| Pairs reported more than once | 509 (0.5%) |
| Pairs with no ownership percentage | 38,287 (35%) |

When a pair is reported more than once, it is one of two things:
- **The same stake reported twice**, usually by two different sources (e.g. 72.2% and 72.24%). Adding them would wrongly double it.
- **Two separate stakes** held by the same owner (e.g. 49% and 51%), which together really are 100%.

### Decisions
| Decision | Why |
|---|---|
| Use current links only | an ownership that has ended can't make a company sanctioned today |
| Leave out companies owning themselves | they would count their own shares |
| One row per owner -> company pair, taking the larger percentage | never double-counts; the rare "two separate stakes" case is slightly undercounted (51% instead of 100%), which rarely changes a 50% result |
| Keep links with no percentage | they can't count towards 50%, but they still show who is connected to whom |
| Keep a low and a high percentage | some sources give ranges (e.g. 75-100%): the low value tells us "sanctioned for sure", the high value "possibly sanctioned" |

In [0]:
WITH pairs AS (
    SELECT owner_id, asset_id,
           count(*) AS n_records,
           min(pct_min) AS pct_lo,
           max(pct_max) AS pct_hi,
           count(pct_min) AS n_with_pct
    FROM pep_dev.silver.ownership
    WHERE is_current AND NOT is_self_owning
    GROUP BY owner_id, asset_id
)

SELECT count(*) as n_pairs, 
count_if(n_records>1) as n_multi,
count_if(n_with_pct=0)  as n_no_pct,
sum(n_records) as n_edges
FROM pairs

In [0]:
  SELECT owner_id, asset_id,
           count(*) AS n_records,
           min(pct_min) AS pct_lo,
           max(pct_max) AS pct_hi,
           count(pct_min) AS n_with_pct
    FROM pep_dev.silver.ownership
    WHERE is_current AND NOT is_self_owning 
    GROUP BY owner_id, asset_id
    HAVING n_records>1;

    SELECT * FROM pep_dev.silver.ownership 
    WHERE owner_id='NK-KgKrVitX9Pv7S4TNvQbZCA' AND asset_id='NK-j2qrz9qzrK2dPNaQkEBZFU'

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.edges_current AS 
SELECT owner_id,
       asset_id,
       owner_type,
       asset_type,
       count(*) AS n_records,
       max(pct_min) AS pct_lo,
       max(pct_max) AS pct_hi
FROM pep_dev.silver.ownership
WHERE is_current
    AND NOT is_self_owning
GROUP BY owner_id,asset_id,owner_type,asset_type;

SELECT * FROM pep_dev.eval.edges_current;


In [0]:
SELECT count(*), count(pct_lo) FROM pep_dev.eval.edges_current

In [0]:
DESCRIBE pep_dev.silver.person

In [0]:
SELECT count(DISTINCT e.owner_id)                         AS person_owners,
       count(DISTINCT CASE WHEN p.person_id IS NOT NULL
                           THEN e.owner_id END)           AS found_in_person
FROM pep_dev.eval.edges_current AS e
LEFT JOIN pep_dev.silver.person AS p
  ON p.person_id = e.owner_id
WHERE e.owner_type = 'Person'

In [0]:
SELECT count_if(is_sanctioned)                                         AS flag_true,
       count_if(array_contains(topics, 'sanction'))                    AS exact_sanction,
       count_if(is_sanctioned AND NOT array_contains(topics, 'sanction')) AS mismatch
FROM pep_dev.silver.person  

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.seeds AS 
SELECT canonical_id AS node_id, entity_type AS node_type
FROM pep_dev.silver.legal_entity
WHERE is_sanctioned
UNION ALL
SELECT person_id AS node_id, 'Person' AS node_type
FROM pep_dev.silver.person
WHERE is_sanctioned


In [0]:
SELECT count(*) FROM pep_dev.eval.seeds


In [0]:
SELECT count(*), entity_type FROM pep_dev.silver.legal_entity
group by entity_type

In [0]:
-- 1. one row per node, no overlap between companies and people
SELECT count(*), count(DISTINCT node_id) FROM pep_dev.eval.seeds;



In [0]:
-- 2. how many seeds actually own something today (where the cascade starts)
SELECT s.node_type, count(DISTINCT s.node_id) AS seeds_that_own
FROM pep_dev.eval.seeds AS s
JOIN pep_dev.eval.edges_current AS e ON e.owner_id = s.node_id
GROUP BY s.node_type;

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.blocked AS   
SELECT node_id,node_type, 0 AS round  
FROM pep_dev.eval.seeds

In [0]:
--ROUND 1 BY HAND

INSERT INTO pep_dev.eval.blocked
SELECT e.asset_id, e.asset_type, 1 
FROM
pep_dev.eval.edges_current AS e
JOIN pep_dev.eval.blocked AS b
ON e.owner_id = b.node_id
WHERE 
e.asset_id NOT IN (SELECT node_id FROM pep_dev.eval.blocked)
GROUP BY e.asset_id, e.asset_type
HAVING sum(e.pct_lo) >= 50

In [0]:
%python

# ------------------------------------------------------------------
# OFAC 50% cascade, as a "repeat until nothing changes" loop
#
# Rule: a company is blocked if blocked owners TOGETHER hold 50% or
#       more of it. Once blocked, it counts as a blocked owner for the
#       companies it owns.
#
# How the loop walks the ownership graph:
#   round 0 = seeds: parties already on a sanctions list
#   round N = companies pushed to 50%+ by owners blocked in earlier rounds
#   Each round reaches one layer further down the ownership chain.
#
# Why it always stops: the blocked list only grows, and a company that's
# already blocked is never added again, so circular ownership
# (A owns B, B owns A) can't make it run forever.
# Each round is written to a Delta table, so Spark's plan doesn't keep
# growing round after round.
# ------------------------------------------------------------------

def run_cascade(target_table: str, pct_col: str, max_rounds: int = 20) -> None:
    """Rebuild target_table from the seeds, then run the cascade on pct_col.
    pct_col = 'pct_lo' -> blocked for sure, 'pct_hi' -> possibly blocked."""

    # Round 0: start (or restart) the blocked list from the seeds
    spark.sql(f"""
        CREATE OR REPLACE TABLE {target_table} AS
        SELECT node_id, node_type, 0 AS round
        FROM pep_dev.eval.seeds
    """)

    for r in range(1, max_rounds + 1):
        res = spark.sql(f"""
            INSERT INTO {target_table}
            SELECT e.asset_id, e.asset_type, :r            -- tag new rows with this round
            FROM pep_dev.eval.edges_current AS e
            JOIN {target_table} AS b
              ON e.owner_id = b.node_id                    -- only links whose OWNER is already blocked
            WHERE e.asset_id NOT IN (SELECT node_id FROM {target_table})  -- skip companies already blocked
            GROUP BY e.asset_id, e.asset_type              -- one bucket per owned company
            HAVING sum(e.{pct_col}) >= 50                  -- blocked owners TOGETHER hold 50%+
        """, args={"r": r})

        n = res.collect()[0]["num_inserted_rows"]          # companies blocked this round
        print(f"{target_table} round {r}: {n} newly blocked")
        if n == 0:                                         # nothing new: the cascade is complete
            break

In [0]:
%python
run_cascade("pep_dev.eval.blocked", "pct_lo")     # blocked for sure

In [0]:
-- DROP TABLE IF EXISTS pep_dev.eval.blocked

In [0]:
SELECT round, count(*) FROM pep_dev.eval.blocked GROUP BY round ORDER BY round

In [0]:
SELECT count(*)                                                  AS we_blocked,
       count_if(array_contains(le.topics, 'sanction.control'))   AS they_also_flag,
       count_if(NOT array_contains(le.topics, 'sanction.control')) AS only_we_flag
FROM pep_dev.eval.blocked AS b
JOIN pep_dev.silver.legal_entity AS le ON le.canonical_id = b.node_id
WHERE b.round > 0